In [2]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import pandas as pd
import anthropic
from dotenv import load_dotenv
load_dotenv()

app = FastAPI(title="Finance AI API", version="1.0")
client = anthropic.Anthropic()

class CommentaryRequest(BaseModel):
    entity: str
    period: str
    comparison_period: str = None

class AnomalyRequest(BaseModel):
    entity: str
    period: str
    threshold: int = 10000

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/api/commentary")
def generate_commentary(req: CommentaryRequest):
    df = pd.read_csv("data/trial_balance.csv")
    data = df[(df["entity"]==req.entity) & (df["period"]==req.period)]
    if data.empty:
        raise HTTPException(status_code=404, detail="No data found")
    
    revenue = data[data["account_type"]=="Income"]["credit"].sum()
    expenses = data[data["account_type"]=="Expense"]["debit"].sum()
    
    msg = client.messages.create(
        model="claude-opus-4-5", max_tokens=512, temperature=0.1,
        system="Finance analyst. Concise board commentary.",
        messages=[{"role": "user", "content":
            f"Commentary for {req.entity} {req.period}: Revenue ${revenue:,.0f}, Expenses ${expenses:,.0f}"}]
    )
    return {"entity": req.entity, "period": req.period,
            "revenue": revenue, "expenses": expenses,
            "commentary": msg.content[0].text}

# In Anaconda Prompt (financeai active): uvicorn day5.api:app --reload
# Docs at: http://localhost:8000/docs

In [4]:
# Add this tool to your existing finops-mcp-server/server.py
# This implements intercompany reconciliation

from mcp.server import Server
from mcp.types import Tool, TextContent
import pandas as pd
import json

# Tool definition (add to your tools list)
INTERCO_RECON_TOOL = Tool(
    name="reconcile_intercompany",
    description="""Reconcile intercompany balances between two entities.
Identifies mismatches between EntityA receivable vs EntityB payable.
Returns structured variance report with per-account gap analysis.""",
    inputSchema={
        "type": "object",
        "properties": {
            "entity_a": {"type": "string", "description": "First entity code"},
            "entity_b": {"type": "string", "description": "Second entity code"},
            "period": {"type": "string", "description": "Period YYYY-MM"},
        },
        "required": ["entity_a", "entity_b", "period"]
    }
)

def handle_reconcile_intercompany(entity_a: str, entity_b: str, period: str) -> dict:
    df = pd.read_csv("data/trial_balance.csv")
    
    # EntityA: what it thinks EntityB owes it (Receivable)
    a_data = df[(df["entity"]==entity_a) & (df["period"]==period)]
    a_receivable = a_data[a_data["account_code"].astype(str)=="1100"]["debit"].sum()
    
    # EntityB: what it thinks it owes EntityA (Payable)
    b_data = df[(df["entity"]==entity_b) & (df["period"]==period)]
    b_payable = b_data[b_data["account_code"].astype(str)=="2001"]["credit"].sum()
    
    gap = a_receivable - b_payable
    return {
        "entity_a": entity_a, "entity_b": entity_b, "period": period,
        "a_receivable": a_receivable, "b_payable": b_payable,
        "gap": gap,
        "reconciled": abs(gap) < 0.01,
        "action_required": "None" if abs(gap) < 0.01 else f"Post correcting entry of ${abs(gap):,.2f}"
    }

In [5]:
import os
import pandas as pd
import json

os.chdir(r"F:\finance-ai-course")

DATA_PATH = r"F:\finance-ai-course\data\trial_balance.csv"

def reconcile_intercompany(entity_a: str, entity_b: str, period: str) -> dict:
    """
    Reconcile intercompany balances between two entities.
    Identifies mismatches between EntityA receivable vs EntityB payable.
    """
    df = pd.read_csv(DATA_PATH)

    # EntityA — what it thinks EntityB owes it (Receivable account 1100)
    a_data = df[(df["entity"] == entity_a) & (df["period"] == period)]
    a_receivable = a_data[a_data["account_code"].astype(str) == "1100"]["debit"].sum()

    # EntityB — what it thinks it owes EntityA (Payable account 2001)
    b_data = df[(df["entity"] == entity_b) & (df["period"] == period)]
    b_payable = b_data[b_data["account_code"].astype(str) == "2001"]["credit"].sum()

    gap = a_receivable - b_payable
    reconciled = abs(gap) < 0.01

    return {
        "entity_a": entity_a,
        "entity_b": entity_b,
        "period": period,
        "a_receivable": a_receivable,
        "b_payable": b_payable,
        "gap": gap,
        "reconciled": str(reconciled),
        "action_required": "None" if reconciled else f"Post correcting entry of ${abs(gap):,.2f}"
    }

# Test it
result = reconcile_intercompany("EntityA", "EntityB", "2025-12")
print(json.dumps(result, indent=2))

{
  "entity_a": "EntityA",
  "entity_b": "EntityB",
  "period": "2025-12",
  "a_receivable": 128400.0,
  "b_payable": 44000.0,
  "gap": 84400.0,
  "reconciled": "False",
  "action_required": "Post correcting entry of $84,400.00"
}


In [6]:
import os

path = r"F:\Finance_ecosystem\finops"
print("Exists:", os.path.exists(path))
print("Contents:", os.listdir(path))

Exists: True
Contents: ['.claude', '.env', '.env.example', '.git', '.gitignore', 'adapters', 'agents', 'api', 'check_esc.py', 'check_stats.py', 'config', 'dashboard', 'db', 'FINOPS_ECOSYSTEM_MASTER_v16.md', 'finops_escalation.db', 'finops_offline.db', 'github_setup.sh', 'ingestion', 'README.md', 'reports', 'requirements.txt', 'requirements.txt.txt', 'run.py', 'tests']


In [7]:
import os

path = r"F:\Finance_ecosystem\finops"

# Look for MCP-related files
for root, dirs, files in os.walk(path):
    # Skip hidden and cache folders
    dirs[:] = [d for d in dirs if d not in ['.git', '__pycache__', '.claude']]
    for file in files:
        if file.endswith('.py'):
            full = os.path.join(root, file)
            print(full.replace(path, ''))

\check_esc.py
\check_stats.py
\run.py
\adapters\accounting_adapter.py
\adapters\market_data_adapter.py
\adapters\__init__.py
\agents\accounting_manager.py
\agents\audit_agents.py
\agents\corp_finance_agents.py
\agents\cost_accountant.py
\agents\financial_controller.py
\agents\fpa_agents.py
\agents\junior_accountant.py
\agents\phase4_orchestrator.py
\agents\revenue_accountant.py
\agents\senior_accountant.py
\agents\tax_agent_tz.py
\agents\tax_agent_us.py
\agents\tax_orchestrator.py
\agents\treasury_agents.py
\agents\__init__.py
\api\escalation.py
\api\main.py
\api\tax_routes.py
\api\__init__.py
\db\store.py
\db\__init__.py
\ingestion\ingestor.py
\ingestion\__init__.py
\reports\reports__init__.py
\reports\tax_pdf_generator.py
\reports\__init__.py
\tests\test_phase4a_agents.py
\tests\test_phase4b_agents.py
\tests\test_session10.py


In [8]:
import os

path = r"C:\Users\🍂\OneDrive - chezsolutions.co.uk\Documents\Claude\Projects\finops-mcp-server"
print("Contents:", os.listdir(path))

Contents: ['.env.example', '.git', '.gitignore', 'README.md', 'requirements.txt', 'server.py']


In [9]:
path = r"C:\Users\🍂\OneDrive - chezsolutions.co.uk\Documents\Claude\Projects\finops-mcp-server"

with open(path + r"\server.py", "r", encoding="utf-8") as f:
    content = f.read()

print(content)

"""
Finance AI Ecosystem MCP Server
Connects Claude Desktop / Claude Code to the Finance AI Ecosystem FastAPI
backend (github.com/Chezhira/finance-ai-pack).

All tools call the FastAPI REST API over HTTP — the MCP server is a pure
wrapper. The ecosystem must be running (uvicorn api.main:app --port 8000)
for tools to work.

Tools — 18 across 3 tiers:

Tier 1 — Core workflow
  ingest_text           → POST /ingest/text
  ingest_email          → POST /ingest/email
  get_suggestions       → GET  /suggestions/{tenant_id}
  get_suggestion        → GET  /suggestions/{tenant_id}/{id}
  decide                → POST /suggestions/{tenant_id}/{id}/decide
  get_stats             → GET  /stats/{tenant_id}
  get_health            → GET  /health

Tier 2 — Department analysis
  analyze_tax           → POST /tax/analyze
  analyze_tax_supervised→ POST /tax/analyze/supervised
  analyze_fpa           → POST /fpa/analyze
  analyze_audit         → POST /audit/analyze
  analyze_treasury      → POST /treasury/a

In [10]:
# Read the current server.py
server_path = r"C:\Users\🍂\OneDrive - chezsolutions.co.uk\Documents\Claude\Projects\finops-mcp-server\server.py"

with open(server_path, "r", encoding="utf-8") as f:
    content = f.read()

# New tool to add
new_tool = '''
# ===========================================================================
# TIER 4 — Direct Analysis (no backend required)
# ===========================================================================

@mcp.tool()
def reconcile_intercompany(
    entity_a: str,
    entity_b: str,
    period: str,
    data_path: Optional[str] = None,
) -> str:
    """
    Reconcile intercompany balances between two entities directly from a
    trial balance CSV. No backend server required — runs against local data.
    Identifies mismatches between EntityA receivable (1100) vs EntityB
    payable (2001) and returns a structured variance report.

    Args:
        entity_a:   First entity code e.g. "EntityA"
        entity_b:   Second entity code e.g. "EntityB"
        period:     Period in YYYY-MM format e.g. "2025-12"
        data_path:  Full path to trial balance CSV. Defaults to env var
                    TRIAL_BALANCE_PATH or the finops default data path.

    Returns:
        Structured intercompany reconciliation report with gap analysis
        and recommended correcting entry.
    """
    try:
        import pandas as pd

        path = data_path or os.getenv("TRIAL_BALANCE_PATH", "data/trial_balance.csv")

        if not os.path.exists(path):
            return f"Data file not found: {path}. Set TRIAL_BALANCE_PATH in .env"

        df = pd.read_csv(path)

        # EntityA receivable — what it thinks EntityB owes it
        a_data = df[(df["entity"] == entity_a) & (df["period"] == period)]
        a_receivable = float(a_data[a_data["account_code"].astype(str) == "1100"]["debit"].sum())

        # EntityB payable — what it thinks it owes EntityA
        b_data = df[(df["entity"] == entity_b) & (df["period"] == period)]
        b_payable = float(b_data[b_data["account_code"].astype(str) == "2001"]["credit"].sum())

        gap = a_receivable - b_payable
        reconciled = abs(gap) < 0.01

        result = {
            "entity_a": entity_a,
            "entity_b": entity_b,
            "period": period,
            "a_receivable": a_receivable,
            "b_payable": b_payable,
            "gap": gap,
            "reconciled": reconciled,
            "risk": "LOW" if reconciled else ("HIGH" if abs(gap) > 50000 else "MEDIUM"),
            "action_required": "None" if reconciled else f"Post correcting entry of ${abs(gap):,.2f}",
            "recommended_entry": "None" if reconciled else (
                f"DR Accounts Payable ({entity_b}) ${abs(gap):,.2f} / "
                f"CR Accounts Receivable ({entity_a}) ${abs(gap):,.2f}"
                if gap > 0 else
                f"DR Accounts Receivable ({entity_a}) ${abs(gap):,.2f} / "
                f"CR Accounts Payable ({entity_b}) ${abs(gap):,.2f}"
            )
        }
        return json.dumps(result, indent=2, default=str)

    except Exception as e:
        return f"Error running reconciliation: {str(e)}"

'''

# Insert before the entry point
entry_point = "# ===========================================================================\n# Entry point"
updated = content.replace(entry_point, new_tool + entry_point)

# Also add pandas to imports comment at top
with open(server_path, "w", encoding="utf-8") as f:
    f.write(updated)

print("Tool added to server.py")

# Verify it's there
with open(server_path, "r", encoding="utf-8") as f:
    new_content = f.read()

if "reconcile_intercompany" in new_content:
    print("Verified: reconcile_intercompany tool is in server.py")
    tool_count = new_content.count("@mcp.tool()")
    print(f"Total tools in server: {tool_count}")
else:
    print("ERROR: tool not found — check manually")

Tool added to server.py
Verified: reconcile_intercompany tool is in server.py
Total tools in server: 20


In [12]:
readme_path = r"C:\Users\🍂\OneDrive - chezsolutions.co.uk\Documents\Claude\Projects\finops-mcp-server\README.md"

with open(readme_path, "r", encoding="utf-8") as f:
    current = f.read()

print(current)

# finops-mcp-server

A Model Context Protocol (MCP) server that connects Claude Desktop and Claude Code to the [Finance AI Ecosystem](https://github.com/Chezhira/finance-accounting-ecosystem) — a multi-agent accounting, tax, FP&A, audit, treasury, and corporate finance system.

Ask Claude to ingest financial data, trigger agent analysis, review suggestion cards, and approve or reject agent recommendations — all from natural language in Claude Desktop.

> **Agents suggest. Humans decide. Nothing posts without approval.**

---

## What it does

Instead of calling the FastAPI endpoints manually, you can ask Claude:

> *"Check if the Finance AI system is online"*
> *"Ingest this invoice text and route it to the right agent"*
> *"Show me all pending suggestions for the default tenant"*
> *"Approve suggestion abc-123"*
> *"Analyse this transaction for Tanzania tax treatment"*
> *"Run this P&L data through the FP&A agents"*
> *"What is the current USD to TZS exchange rate?"*
> *"List all open

In [13]:
readme_path = r"C:\Users\🍂\OneDrive - chezsolutions.co.uk\Documents\Claude\Projects\finops-mcp-server\README.md"

with open(readme_path, "r", encoding="utf-8") as f:
    content = f.read()

# Update tool count in title
content = content.replace("Tools — 19 across 3 tiers", "Tools — 20 across 4 tiers")

# Add Tier 4 section after Tier 3 table
tier4 = """
### Tier 4 — Direct Analysis (no backend required)

| Tool | Description |
|------|-------------|
| `reconcile_intercompany` | Reconcile intercompany balances between two entities directly from a trial balance CSV. Identifies AR/AP mismatches, calculates the gap, and returns a recommended correcting journal entry. No backend server required. |

"""

content = content.replace("---\n\n## Prerequisites", tier4 + "---\n\n## Prerequisites")

with open(readme_path, "w", encoding="utf-8") as f:
    f.write(content)

print("README updated")

# Verify
with open(readme_path, "r", encoding="utf-8") as f:
    updated = f.read()

if "reconcile_intercompany" in updated and "20 across 4 tiers" in updated:
    print("Verified: tool count updated to 20, Tier 4 added")
else:
    print("ERROR: check manually")

README updated
Verified: tool count updated to 20, Tier 4 added
